In [ ]:
# Wurde auf einem Laptop mit GPU ausgeführt. Embeddings wurden anschließend in data gespeichert.

import pandas as pd
import os
import numpy as np

import torch #Für Arbeit mit neuronalen Netzen
from esm.models.esmc import ESMC
from esm.utils.constants.models import ESMC_600M
from esm.tokenization import EsmSequenceTokenizer
# EsmSequenceTokenizer ist eine Objektklasse, die Methoden wie .encode() und .decode() hat,
# um zwischen Sequenz und Aminosäure-Tokens zu unterscheiden
from esm.utils import encoding
# encoding ist ein Modul (.py-Datei), das Funktionen wie tokenize_sequence(seq=str, tokenizer=EsmSequenceTokenizer, ...) enthält

esm = ESMC.from_pretrained("esmc_600m").to('cuda')
# Lädt das trainierte (d.h. z.B. optimierte weights und biases durch Minimuerung der Loss-Function) ESMC-Modell mit dem Befehl, das Modell auf der GPU auszuführen
tokenizer = EsmSequenceTokenizer()
# Lädt den Tokenizer als Objekt einer Klasse mit Methoden wie .encode() und .decode(), um zwischen AS-Identität und zugehöriger Token-ID zu übersetzen

In [ ]:
regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
embed_path = "data/embeddings/scalop"
df = pd.read_csv("data/ab_ag_scalop.tsv", sep="\t").dropna(subset = regions)

fab_lists = df[["pdb", "HChain", "Lchain"] + regions].values.tolist()

for fab_list in fab_lists:
    pdb = fab_list[0]
    hchain = fab_list[1]
    lchain = fab_list[2]
    seqs = fab_list[3:8]


    for region, seq in zip(regions, seqs):
        chain_type = region[4] #z.B. "H" aus "SEQ_H2" oder "L" aus "SEQ_L3"
        if chain_type == "H":
            chain_id = hchain
        else:
            chain_id = lchain

        embed_file_name = f"{pdb}_{chain_id}_{chain_type}_{region}_chothia.npy"

        with torch.no_grad():
        # no_grad heißt: keine Backpropagation für Anpassung der weights und bias um loss-function zu minimieren (= Modell trainieren), sondern neuronales Netz nur "vorwärts" laufen lassen
            tokenized = encoding.tokenize_sequence(seq, tokenizer, add_special_tokens=True).to('cuda')
            # encoding: ein Modul aus dem Paket esm.utils
            # tokenize_sequence: Funktion aus dem Modul encoding: Wandelt AS-Sequenz (str) in das Input-Layer des neuralen Netzes um, indem es jeder AS-Identität (str) eine Token-ID (num) zuweist
            # add_special_tokens=True: fügt am Anfang und Ende der Token-Sequenz je ein Sondertoken ein, um Anfang und Ende für das Modell sichtbar zu machen
            pred = esm.forward(tokenized.unsqueeze(0))
            # tokenized.unsqueeze(0) fügt dem Input-Vektor tokenized eine zusätzliche Dimension hinzu, die der batch_size(= Anzahl der gleichzeitig untersuchten Proteine) entspricht.
            # tokenized wird formal zu einer Matrix, obwohl die zusätzliche Dimension leer ist, weil bei uns: batch_size = 1
            # --> tokenized.unsequeeze(0).shape = [Sequenzlänge, batch_size = 1]
            # notwendig, weil esm.forward() eine solche Matrix erwartet
            # esm.forward(): "schickt" die Input-Matrix durch das Transformer-Netzwerk (d.h. führt Matrix-Multiplikation mit den beim Training eingestellten weights und biases durch)
            # --> pred ist eine Matrix aus den logits des Output-Layers (Reihen entsprechen einzelnen AS in der Sequenzen, mehrere Spalten pro AS repräsentieren eine AS im Ouuput-Layer des Transformers)
            embeddings = pred.embeddings.to('cpu').squeeze()
            # pred.embeddings: prozessiert die Matrix (WIE?) und speichert sie als embeddings ab
            # .to('cpu'): embeddings wird wieder in der cpu gespeichert, da numpy hier arbeitet
            # .squeeze: .unsqueeze() wird wieder rückgängig gemacht
            embed_as_arr = embeddings.float().detach().numpy()
            # .float():Klassen der Einträge der Logits werden in floats umgewandelt. GPU hat vorher mit bfloat16 gearbeitet
            # .detach(): Informationen über das zugrungdeliegende Transformer-Netz wird verworfen, da wir nicht an Backpropagation interessiert sind
            # .numpy(): wandelt embeddings von der Klasse tensor aus PyTorch in ein array aus NumPy um
            embed_as_arr = embed_as_arr[1:-1,:]
            # Entfernt die SpecialTokens, die in der ersten Code-Zeile zugefügt wurden
            # --> entfernt erste und letzte Reihe des 2D-arrays, alle Spalten bleiben erhalten
            if not os.path.exists(embed_path):
                os.makedirs(embed_path)
            np.save(os.path.join(embed_path, embed_file_name), embed_as_arr)
            # Speichert das NumPy-Array ab'''

In [ ]:
regions = ["CDR_H1", "CDR_H2", "CDR_L1", "CDR_L2", "CDR_L3"]
embed_path = "data/embeddings/chothia"
df = pd.read_csv("data/ab_ag_chothia.tsv", sep="\t").dropna(subset = regions)

fab_lists = df[["pdb", "HChain", "Lchain"] + regions].values.tolist()

for fab_list in fab_lists:
    pdb = fab_list[0]
    hchain = fab_list[1]
    lchain = fab_list[2]
    seqs = fab_list[3:8]


    for region, seq in zip(regions, seqs):
        chain_type = region[4] #z.B. "H" aus "CDR_H2" oder "L" aus "CDR_L3"
        if chain_type == "H":
            chain_id = hchain
        else:
            chain_id = lchain

        embed_file_name = f"{pdb}_{chain_id}_{chain_type}_{region}_chothia.npy"

        with torch.no_grad():
        # no_grad heißt: keine Backpropagation für Anpassung der weights und bias um loss-function zu minimieren (= Modell trainieren), sondern neuronales Netz nur "vorwärts" laufen lassen
            tokenized = encoding.tokenize_sequence(seq, tokenizer, add_special_tokens=True).to('cuda')
            # encoding: ein Modul aus dem Paket esm.utils
            # tokenize_sequence: Funktion aus dem Modul encoding: Wandelt AS-Sequenz (str) in das Input-Layer des neuralen Netzes um, indem es jeder AS-Identität (str) eine Token-ID (num) zuweist
            # add_special_tokens=True: fügt am Anfang und Ende der Token-Sequenz je ein Sondertoken ein, um Anfang und Ende für das Modell sichtbar zu machen
            pred = esm.forward(tokenized.unsqueeze(0))
            # tokenized.unsqueeze(0) fügt dem Input-Vektor tokenized eine zusätzliche Dimension hinzu, die der batch_size(= Anzahl der gleichzeitig untersuchten Proteine) entspricht.
            # tokenized wird formal zu einer Matrix, obwohl die zusätzliche Dimension leer ist, weil bei uns: batch_size = 1
            # --> tokenized.unsequeeze(0).shape = [Sequenzlänge, batch_size = 1]
            # notwendig, weil esm.forward() eine solche Matrix erwartet
            # esm.forward(): "schickt" die Input-Matrix durch das Transformer-Netzwerk (d.h. führt Matrix-Multiplikation mit den beim Training eingestellten weights und biases durch)
            # --> pred ist eine Matrix aus den logits des Output-Layers (Reihen entsprechen einzelnen AS in der Sequenzen, mehrere Spalten pro AS repräsentieren eine AS im Ouuput-Layer des Transformers)
            embeddings = pred.embeddings.to('cpu').squeeze()
            # pred.embeddings: prozessiert die Matrix (WIE?) und speichert sie als embeddings ab
            # .to('cpu'): embeddings wird wieder in der cpu gespeichert, da numpy hier arbeitet
            # .squeeze: .unsqueeze() wird wieder rückgängig gemacht
            embed_as_arr = embeddings.float().detach().numpy()
            # .float():Klassen der Einträge der Logits werden in floats umgewandelt. GPU hat vorher mit bfloat16 gearbeitet
            # .detach(): Informationen über das zugrungdeliegende Transformer-Netz wird verworfen, da wir nicht an Backpropagation interessiert sind
            # .numpy(): wandelt embeddings von der Klasse tensor aus PyTorch in ein array aus NumPy um
            embed_as_arr = embed_as_arr[1:-1,:]
            # Entfernt die SpecialTokens, die in der ersten Code-Zeile zugefügt wurden
            # --> entfernt erste und letzte Reihe des 2D-arrays, alle Spalten bleiben erhalten
            if not os.path.exists(embed_path):
                os.makedirs(embed_path)
            np.save(os.path.join(embed_path, embed_file_name), embed_as_arr)
            # Speichert das NumPy-Array ab'''

In [ ]:
'''df = pd.read_csv("data/ab_ag_scalop.tsv")
regions = ["H1", "H2", "L1", "L2", "L3"]

df_rows = df[["PDB_ID", "CDR", "CDR_Sequence", "Canonical_Form", "Structure"]].values.tolist()
seen = defaultdict(int)

for df_row in df_rows:
    pdb = df_row[0]
    region = df_row[1]
    sequence = df_row[3]
    canonical_form = df_row[4]

    seen[(pdb, region)] += 1
    counter = seen[(pdb, region)]

    embed_path = "data/embeddings/scalop"
    embed_filepdb = f"{pdb}_{counter}_CDR_{region}_scalop.npy"

    tokenized = encoding.tokenize_sequence(sequence, tokenizer, add_special_tokens=True).to('cuda')
    # encoding: ein Modul aus dem Paket esm.utils
    # tokenize_sequence: Funktion aus dem Modul encoding: Wandelt AS-Sequenz (str) in das Input-Layer des neuralen Netzes um, indem es jeder AS-Identität (str) eine Token-ID (num) zuweist
    # add_special_tokens=True: fügt am Anfang und Ende der Token-Sequenz je ein Sondertoken ein, um Anfang und Ende für das Modell sichtbar zu machen
    pred = esm.forward(tokenized.unsqueeze(0))
    # tokenized.unsqueeze(0) fügt dem Input-Vektor tokenized eine zusätzliche Dimension hinzu, die der batch_size(= Anzahl der gleichzeitig untersuchten Proteine) entspricht.
    # tokenized wird formal zu einer Matrix, obwohl die zusätzliche Dimension leer ist, weil bei uns: batch_size = 1
    # --> tokenized.unsequeeze(0).shape = [Sequenzlänge, batch_size = 1]
    # notwendig, weil esm.forward() eine solche Matrix erwartet
    # esm.forward(): "schickt" die Input-Matrix durch das Transformer-Netzwerk (d.h. führt Matrix-Multiplikation mit den beim Training eingestellten weights und biases durch)
    # --> pred ist eine Matrix aus den logits des Output-Layers (Reihen entsprechen einzelnen AS in der Sequenzen, mehrere Spalten pro AS repräsentieren eine AS im Ouuput-Layer des Transformers)
    embeddings = pred.embeddings.to('cpu').squeeze()
    # pred.embeddings: prozessiert die Matrix (WIE?) und speichert sie als embeddings ab
    # .to('cpu'): embeddings wird wieder in der cpu gespeichert, da numpy hier arbeitet
    # .squeeze: .unsqueeze() wird wieder rückgängig gemacht
    embed_as_arr = embeddings.float().detach().numpy()
    # .float():Klassen der Einträge der Logits werden in floats umgewandelt. GPU hat vorher mit bfloat16 gearbeitet
    # .detach(): Informationen über das zugrungdeliegende Transformer-Netz wird verworfen, da wir nicht an Backpropagation interessiert sind
    # .numpy(): wandelt embeddings von der Klasse tensor aus PyTorch in ein array aus NumPy um
    embed_as_arr = embed_as_arr[1:-1,:]
    # Entfernt die SpecialTokens, die in der ersten Code-Zeile zugefügt wurden
    # --> entfernt erste und letzte Reihe des 2D-arrays, alle Spalten bleiben erhalten
    if not os.path.exists(embed_path):
        os.makedirs(embed_path)
    np.save(os.path.join(embed_path, embed_filepdb), embed_as_arr)
    # Speichert das NumPy-Array ab'''

In [ ]:
'''df = pd.read_csv("data/ab_ag_chothia.tsv", sep="\t")
regions = ["CDR_H1", "CDR_H2", "CDR_L1", "CDR_L2", "CDR_L3"]

seen = defaultdict(int)

for region in regions:
    to_process = [
       (pdb, seq)
       for pdb, seq in zip(df.pdb, df[region])
    ]
    with torch.no_grad():
    # no_grad heißt: keine Backpropagation für Anpassung der weights und bias um loss-function zu minimieren (= Modell trainieren), sondern neuronales Netz nur "vorwärts" laufen lassen
        for pdb, seq in to_process:
            
            seen[(pdb, region)] += 1
            counter = seen[(pdb, region)]
            
            embed_path = "data/embeddings/chothia"
            embed_filepdb = f"{pdb}_{counter}_{region}_chothia.npy"

            tokenized = encoding.tokenize_sequence(seq, tokenizer, add_special_tokens=True).to('cuda')
            # encoding: ein Modul aus dem Paket esm.utils
            # tokenize_sequence: Funktion aus dem Modul encoding: Wandelt AS-Sequenz (str) in das Input-Layer des neuralen Netzes um, indem es jeder AS-Identität (str) eine Token-ID (num) zuweist
            # add_special_tokens=True: fügt am Anfang und Ende der Token-Sequenz je ein Sondertoken ein, um Anfang und Ende für das Modell sichtbar zu machen
            pred = esm.forward(tokenized.unsqueeze(0))
            # tokenized.unsqueeze(0) fügt dem Input-Vektor tokenized eine zusätzliche Dimension hinzu, die der batch_size(= Anzahl der gleichzeitig untersuchten Proteine) entspricht.
            # tokenized wird formal zu einer Matrix, obwohl die zusätzliche Dimension leer ist, weil bei uns: batch_size = 1
            # --> tokenized.unsequeeze(0).shape = [Sequenzlänge, batch_size = 1]
            # notwendig, weil esm.forward() eine solche Matrix erwartet
            # esm.forward(): "schickt" die Input-Matrix durch das Transformer-Netzwerk (d.h. führt Matrix-Multiplikation mit den beim Training eingestellten weights und biases durch)
            # --> pred ist eine Matrix aus den logits des Output-Layers (Reihen entsprechen einzelnen AS in der Sequenzen, mehrere Spalten pro AS repräsentieren eine AS im Ouuput-Layer des Transformers)
            embeddings = pred.embeddings.to('cpu').squeeze()
            # pred.embeddings: prozessiert die Matrix (WIE?) und speichert sie als embeddings ab
            # .to('cpu'): embeddings wird wieder in der cpu gespeichert, da numpy hier arbeitet
            # .squeeze: .unsqueeze() wird wieder rückgängig gemacht
            embed_as_arr = embeddings.float().detach().numpy()
            # .float():Klassen der Einträge der Logits werden in floats umgewandelt. GPU hat vorher mit bfloat16 gearbeitet
            # .detach(): Informationen über das zugrungdeliegende Transformer-Netz wird verworfen, da wir nicht an Backpropagation interessiert sind
            # .numpy(): wandelt embeddings von der Klasse tensor aus PyTorch in ein array aus NumPy um
            embed_as_arr = embed_as_arr[1:-1,:]
            # Entfernt die SpecialTokens, die in der ersten Code-Zeile zugefügt wurden
            # --> entfernt erste und letzte Reihe des 2D-arrays, alle Spalten bleiben erhalten
            if not os.path.exists(embed_path):
                os.makedirs(embed_path)
            np.save(os.path.join(embed_path, embed_filepdb), embed_as_arr)
            # Speichert das NumPy-Array ab'''